# Classificazione del Diabete

Obiettivo: classificare ogni paziente in due classi:
- `0` = diabete **non** presente
- `1` = diabete **presente**

Dataset: [Diabetes Prediction Dataset (Kaggle)](https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset?select=diabetes_prediction_dataset.csv)

Variabili disponibili: `gender`, `age`, `hypertension`, `heart_disease`, `smoking_history`, `bmi`, `HbA1c_level`, `blood_glucose_level`, `diabetes` (target).

In [ ]:
# --- IMPORT LIBRERIE
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

root = Path.cwd()
file_path = root / "diabetes_prediction_dataset.csv"

## Caricamento e prima esplorazione

In [ ]:
df = pd.read_csv(file_path)

print(f"Shape: {df.shape}")
print(f"\nInfo:")
df.info()

In [ ]:
df.head(10)

In [ ]:
df.describe()

## Pulizia dei dati

In [ ]:
# Valori nulli
print("Valori nulli per colonna:")
print(df.isnull().sum())

# Duplicati
n_dup = df.duplicated().sum()
print(f"\nRighe duplicate: {n_dup} ({n_dup / len(df) * 100:.2f}%)")

# Rimozione duplicati
df = df.drop_duplicates()
print(f"Shape dopo rimozione duplicati: {df.shape}")

## Encoding delle variabili categoriche

**`smoking_history`** — contiene anche valori `"No Info"` (~35% dei dati): non li sostituiamo, lasciamo che il modello ne apprenda il pattern.

**`gender`** — il valore `"Other"` rappresenta solo lo 0.018% dei dati: lo rimuoviamo prima dell'encoding.

In [ ]:
print("=== smoking_history ===")
print(df["smoking_history"].value_counts(normalize=True).mul(100).round(2))

print("\n=== gender ===")
print(df["gender"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# Rimuovo il valore "Other" da gender
df = df[df["gender"] != "Other"]

# LabelEncoding
le_gender  = LabelEncoder()
le_smoking = LabelEncoder()

df["gender"]          = le_gender.fit_transform(df["gender"])
df["smoking_history"] = le_smoking.fit_transform(df["smoking_history"])

print("Mapping gender:",  dict(zip(le_gender.classes_,  le_gender.transform(le_gender.classes_))))
print("Mapping smoking:", dict(zip(le_smoking.classes_, le_smoking.transform(le_smoking.classes_))))

## Split Train / Test

In [ ]:
X = df.drop(columns=["diabetes"])
y = df["diabetes"]

print(f"Features: {X.shape}  |  Target: {y.shape}")
print(f"\nDistribuzione classi:\n{y.value_counts(normalize=True).round(3)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,   # preserva le proporzioni (≈91% / 9%)
)

## Modello — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",   # compensa lo sbilanciamento delle classi
)
rf.fit(X_train, y_train)

# Accuracy su train e test
print(f"Accuracy TRAIN = {rf.score(X_train, y_train):.4f}")
print(f"Accuracy TEST  = {rf.score(X_test,  y_test):.4f}")

## Valutazione

In [ ]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["No diabete", "Diabete"]))

print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
# 0.50 → inutile  |  0.85 → buono  |  0.95 → ottimo  |  1.00 → sospetto (overfitting)

In [ ]:
# Matrice di confusione
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots()
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No diabete", "Diabete"]).plot(ax=ax, cmap="Blues")
ax.set_title("Matrice di confusione")
plt.tight_layout()
# plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

## Predizione su un nuovo paziente

In [ ]:
nuovo_paziente = pd.DataFrame([{
    "gender":              le_gender.transform(["Male"])[0],
    "age":                 46,
    "hypertension":        0,
    "heart_disease":       0,
    "smoking_history":     le_smoking.transform(["former"])[0],
    "bmi":                 30.5,
    "HbA1c_level":         6.8,
    "blood_glucose_level": 145,
}])

prob = rf.predict_proba(nuovo_paziente)[0][1]
pred = rf.predict(nuovo_paziente)[0]

print(f"Probabilità diabete : {prob:.2%}")
print(f"Classe predetta      : {'Diabete' if pred == 1 else 'No diabete'}")